# 실습 — 사내규정 문서를 불러오고 잘게 쪼개기

**현대경제연구소 사내 헬프데스크 챗봇 '현경봇' 만들기 **

> ⚠️ 이 노트북의 규정은 **교육용으로 만든 가상 규정**입니다.

## 시나리오

지난 시간에 '현경봇'은 규정을 **통째로** 받아 정확하게 답했습니다.
그런데 인사팀이 규정 문서를 5개로 늘리고 내용도 훨씬 자세해졌습니다.

| 파일 | 내용 | 담당 |
|------|------|------|
| `01_faq.md` | 사내 헬프데스크 FAQ (입사 첫날, 회의실, 연락처) | 공통 |
| `02_leave.md` | 근무·휴가 (시차출퇴근, 재택, 연차, 경조, 육아) | 인사팀 |
| `03_travel.md` | 출장·경비 (출장비 표, 교통비, 법인카드) | 재무팀 |
| `04_security.md` | 정보보안 (보안등급, 엠바고, 외부 AI) | IT팀 |
| `05_welfare.md` | 복리후생 (자기계발비, 건강검진, 주차) | 인사팀·총무팀 |

신입 김연구 씨가 "재택근무 돼요?" 한 줄 물을 때마다 5개 문서를 다 넣을 순 없습니다.
그래서 오늘은 문서를 **불러오고(Load)**, **잘게 쪼개고(Split)**, **꼬리표(metadata)** 를 답니다.

| 단계 | 내용 |
|------|------|
| 1️⃣ 이론 | 왜 불러오고 왜 쪼개나? |
| 2️⃣ 기본 | 파일 불러오기 + 쪼개기 |
| 3️⃣ 응용 | 쪼개는 크기·겹침을 바꿔보기 |
| 4️⃣ 실습 | 사내규정 5개를 쪼개고 분류·담당부서 꼬리표 달기 |
| 5️⃣ 미션 | 김연구 씨 질문에 맞는 청크 찾기 + 제목 기준으로 쪼개기 |

## 📝 실습 방법

- 코드 속 `____` (밑줄 4개)를 알맞은 코드로 바꾼 뒤 실행하세요.
- 💡 막히면 **위니브 카페 노트북([Day5_2])** 의 같은 셀을 참고하세요.

# 1️⃣ 단계 — 이론

## 지난 시간의 숙제

규정 3개를 통째로 넣었더니 토큰이 몇 배로 늘었습니다.
이번엔 규정이 5개이고 더 깁니다. 매 질문마다 통째로 넣으면 **비싸고 느립니다.**

> 🗝️ 결론: **"통째로 말고, 질문에 필요한 조항만 주자."**

## 준비 ① 불러오기 (Document Loader)

규정은 지금 `.md` **파일**입니다. 프로그램이 다루려면 파일을 읽어 들여야 합니다.

> 💡 비유: 종이 규정집을 **스캔**해서 컴퓨터에 넣는 것. Document Loader가 스캐너입니다.

불러온 결과 = **본문**(`page_content`) + **꼬리표**(`metadata`, 출처 정보)

> 🔑 꼬리표 덕분에 나중에 "(근거: 근무·휴가 규정 / 문의: 인사팀)"처럼 출처를 붙이고,
> "보안 질문엔 보안 규정만 검색"할 수 있습니다.

## 준비 ② 쪼개기 (Text Splitter)

규정집 전체 대신 **조항 단위의 작은 조각(청크, Chunk)** 으로 자릅니다.
그래야 "재택근무 조항"만 콕 집어서 AI에게 줄 수 있습니다.

## 두 개의 손잡이

- **chunk_size**: 한 조각의 크기(글자 수). 너무 작으면 문맥이 끊기고, 너무 크면 다시 비싸집니다.
- **chunk_overlap**: 이웃 조각끼리 겹치는 글자 수. "최대 5일" 같은 핵심 정보가 경계에서 잘리지 않게 합니다.

# 2️⃣ 단계 — 파일 불러오고 쪼개보기

### 셀 1 — 도구 설치

In [ ]:
!pip install -q langchain langchain-community langchain-text-splitters python-dotenv

### 셀 2 — 연습용 파일 만들기

사내 안내문 한 장을 파일로 만들어 봅니다.

In [ ]:
text = """현대경제연구소 사내 안내
사내식당 중식은 11시 30분부터 13시까지입니다.
5층 자료실은 평일 9시부터 20시까지 운영합니다.
IT헬프데스크 내선번호는 1234입니다."""

# TODO: 파일을 '쓰기' 모드로 열기 (힌트: 쓰기 모드는 한 글자)
with open("hri_sample.txt", "____", encoding="utf-8") as f:
    f.write(text)

print("hri_sample.txt 파일을 만들었습니다.")

### 셀 3 — 파일 불러오기 (TextLoader)

In [ ]:
from langchain_community.document_loaders import TextLoader

# TODO: 셀 2에서 만든 파일 이름을 넣으세요
loader = TextLoader("____", encoding="utf-8")

# TODO: 파일을 읽어들이는 메서드
docs = loader.____()

doc = docs[0]
print("본문:", doc.____)      # TODO: 본문이 들어있는 속성
print()
print("꼬리표:", doc.____)    # TODO: 꼬리표(출처)가 들어있는 속성

> 💡 꼬리표에 `{'source': 'hri_sample.txt'}`가 자동으로 붙어 있습니다.

### 셀 4 — 문장 쪼개기 (RecursiveCharacterTextSplitter)

재택근무 조항 문장을 약 30자 조각으로, 이웃과 5자 겹치게 자릅니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

long_text = ("재택근무는 주 2회까지 가능합니다. 전날 17시까지 신청해야 합니다. "
             "팀장 승인이 필요합니다. 입사 3개월 미만은 재택근무를 할 수 없습니다.")

# TODO: 한 조각 30자, 겹침 5자
splitter = RecursiveCharacterTextSplitter(chunk_size=____, chunk_overlap=____)

# TODO: '글자열'을 쪼개는 메서드
pieces = splitter.____(long_text)

print("조각 개수:", len(pieces))
for i, p in enumerate(pieces):
    print(f"[{i}] {p}")

### 셀 5 — 불러온 문서를 바로 쪼개기

`split_documents`를 쓰면 **꼬리표가 조각마다 복사**됩니다.

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=30, chunk_overlap=5)

# TODO: 셀 3의 docs(Document 리스트)를 쪼개는 메서드
chunks = splitter.____(docs)

print("조각 개수:", len(chunks))
for c in chunks:
    print("본문:", c.page_content, "| 꼬리표:", c.metadata)

> 📌 **불러오기**(`loader.load()`) → **쪼개기**(`splitter.split_documents()`). 오늘의 핵심 두 줄입니다.

# 3️⃣ 단계 — 크기와 겹침 바꿔보기

### 셀 6 — chunk_size에 따라 조각 수 비교

In [ ]:
base_text = ("연차는 사용 3영업일 전까지 그룹웨어로 신청합니다. "
             "3일 이상 연속 사용 시 2주 전까지 신청해야 합니다. "
             "반차는 오전과 오후 단위로 사용합니다. 반반차는 없습니다. ") * 5

# TODO: 30, 100, 300 세 가지 크기로 비교해 보세요
for size in [____, ____, ____]:
    sp = RecursiveCharacterTextSplitter(chunk_size=____, chunk_overlap=0)   # TODO: 반복 변수 사용
    pieces = sp.split_text(base_text)
    print(f"chunk_size={size:>3} → 조각 {len(pieces)}개")

> 💡 작을수록 조각이 많아지고, 클수록 적어집니다.

> ✍️ 내가 고른 적당한 크기와 이유: 

### 셀 7 — overlap 있을 때 vs 없을 때

이월 연차 조항입니다. 경계에서 "최대 5일"이나 "3월 31일"이 잘리는지 보세요.

In [ ]:
text = "미사용 연차는 다음 해로 최대 5일까지 이월할 수 있으며 다음 해 3월 31일까지 사용해야 합니다."

print("───[ overlap=0 ]───")
for p in RecursiveCharacterTextSplitter(chunk_size=25, chunk_overlap=____).split_text(text):   # TODO: 0
    print(" ", p)

print("\n───[ overlap=10 ]───")
for p in RecursiveCharacterTextSplitter(chunk_size=25, chunk_overlap=____).split_text(text):   # TODO: 10
    print(" ", p)

> 💡 overlap이 있으면 다음 조각이 앞 조각의 끝을 다시 포함해서, "이월"과 "3월 31일"이 함께 남을 가능성이 커집니다.

# 4️⃣ 단계 — 사내규정 5개 쪼개기

### 셀 8 — 규정 문서 5개를 파일로 저장

인사팀에서 받은 규정 문서 5개를 `hri_docs/` 폴더에 저장합니다.

In [ ]:
import os

# 현대경제연구소 사내규정 문서 5개 (※ 교육용 가상 규정)
documents = {
    "01_faq.md": """# 현대경제연구소 사내 헬프데스크 자주 묻는 질문 (FAQ)

※ 교육용 가상 문서입니다. 실제 기관의 규정과 무관합니다.

사내 헬프데스크 '현경봇'이 자주 받는 질문을 모았습니다.
자세한 내용은 각 규정 문서를 확인해 주세요.

---

### Q. 입사 첫날에는 무엇을 해야 하나요?
A. 오전 9시까지 본관 3층 인사팀으로 오시면 사원증, 노트북, 보안 USB를 지급합니다.
오후에는 정보보안 교육(2시간)을 반드시 이수해야 사내 시스템 계정이 활성화됩니다.

### Q. 그룹웨어 계정은 언제 생기나요?
A. 정보보안 교육 이수 후 당일 18시까지 생성됩니다. 초기 비밀번호는 문자로 안내되며,
최초 로그인 시 반드시 변경해야 합니다.

### Q. 연차·재택근무 신청은 어디서 하나요?
A. 그룹웨어 > 근태관리 메뉴에서 신청합니다. 신청 기한은 근무·휴가 규정을 확인해 주세요.

### Q. 출장비 정산은 어떻게 하나요?
A. 그룹웨어 > 경비정산 메뉴에서 영수증 사진을 첨부해 제출합니다.
출장 복귀 후 7영업일 이내에 정산해야 합니다.

### Q. 노트북이 고장 났어요.
A. IT헬프데스크(내선 1234)로 연락하시면 대체 노트북을 당일 지급합니다.
개인 PC로 업무 자료를 옮기는 것은 금지되어 있습니다.

### Q. 사내 도서관은 어떻게 이용하나요?
A. 본관 5층 자료실은 평일 09:00~20:00 운영합니다. 연구 자료는 1인 10권까지
2주간 대출할 수 있으며, 대외비 보고서는 열람만 가능합니다.

### Q. 외부 기고나 강연 요청이 왔어요.
A. 외부 활동은 사전에 본부장 승인과 홍보팀 협의가 필요합니다.
미발표 연구 결과나 전망 수치는 언급할 수 없습니다.

### Q. 명함은 어떻게 신청하나요?
A. 그룹웨어 > 총무 > 명함신청 메뉴에서 신청하면 5영업일 이내 자리로 배송됩니다.

### Q. 회의실 예약은 어떻게 하나요?
A. 그룹웨어 > 시설예약에서 예약합니다. 1회 최대 3시간까지 예약할 수 있으며,
대회의실(30인)은 팀장 이상만 예약할 수 있습니다.

### Q. 각 부서 연락처를 알려주세요.
A. 인사팀(내선 2100), 재무팀(내선 2200), IT헬프데스크(내선 1234),
총무팀(내선 2300), 홍보팀(내선 2400)입니다.
""",

    "02_leave.md": """# 현대경제연구소 근무·휴가 규정

※ 교육용 가상 문서입니다. 실제 기관의 규정과 무관합니다.

본 규정은 현대경제연구소 전 직원(계약직 포함)에게 적용됩니다. 담당: 인사팀

## 제1조 근무시간

- 기본 근무시간은 09:00~18:00 이며, 점심시간은 12:00~13:00 입니다.
- **시차출퇴근제**: 08:00~10:00 사이에서 출근 시각을 선택하고, 하루 8시간을 근무합니다.
- 출근 시각은 월 단위로 정하며, 변경하려면 전월 25일까지 그룹웨어에 신청합니다.
- 연장근무는 사전에 팀장 승인을 받아야 하며, 주 12시간을 넘을 수 없습니다.

## 제2조 재택근무

- 재택근무는 **주 2회까지** 가능합니다.
- 전날 17시까지 그룹웨어에서 신청하고 팀장 승인을 받아야 합니다.
- **입사 3개월 미만 직원은 재택근무를 할 수 없습니다.**
- 재택근무 중에도 09:00~16:00(코어타임)에는 메신저 응답이 가능해야 합니다.
- 대외비 자료는 재택근무 중 열람할 수 없습니다. (정보보안 규정 참고)

## 제3조 연차휴가

| 근속 기간 | 연차 일수 |
|-----------|-----------|
| 1년 미만 | 1개월 개근 시 1일 (최대 11일) |
| 1년 이상 3년 미만 | 15일 |
| 3년 이상 5년 미만 | 16일 |
| 5년 이상 7년 미만 | 17일 |
| 7년 이상 | 2년마다 1일 가산 (최대 25일) |

- 반차(0.5일)는 오전(09:00~13:00), 오후(14:00~18:00) 단위로 사용합니다.
- **반반차(2시간 단위)는 없습니다.**
- 연차는 사용 **3영업일 전**까지 그룹웨어로 신청합니다.
- **3일 이상 연속** 사용 시 **2주 전**까지 신청해야 합니다.
- 긴급한 경우 당일 09시 전 팀장에게 연락하고, 복귀 후 2영업일 이내 사후 신청합니다.

## 제4조 미사용 연차

- 미사용 연차는 다음 해로 **최대 5일까지 이월**할 수 있습니다.
- 이월된 연차는 **다음 해 3월 31일까지** 사용해야 하며, 기한이 지나면 소멸합니다.
- 연차사용촉진제를 시행하므로, 이월분을 제외한 미사용 연차는 금전 보상하지 않습니다.
- 퇴직 시 남은 연차는 퇴직 정산 시 수당으로 지급합니다.

## 제5조 경조휴가 (유급)

| 구분 | 휴가 | 경조금 |
|------|------|--------|
| 본인 결혼 | 5일 | 50만원 |
| 자녀 결혼 | 1일 | 30만원 |
| 배우자 출산 | 10일 | 20만원 |
| 부모·배우자 부모 사망 | 5일 | 50만원 |
| 조부모·배우자 조부모 사망 | 3일 | 20만원 |
| 형제자매 사망 | 2일 | 10만원 |

- 경조휴가는 주말·공휴일을 제외하고 계산합니다.
- 배우자 출산휴가는 출산일로부터 90일 이내에 사용해야 하며, 1회 분할 사용이 가능합니다.
- 경조금은 증빙서류(청첩장, 가족관계증명서 등) 제출 후 다음 달 급여와 함께 지급합니다.

## 제6조 병가

- 유급 병가는 연 5일입니다. 이를 초과하면 연차에서 차감합니다.
- 3일 이상 연속 병가 시 진단서를 제출해야 합니다.
- 업무상 부상·질병은 산재 절차에 따르며, 인사팀에 즉시 알려야 합니다.

## 제7조 육아 지원

- 육아휴직: 만 8세 이하 자녀 1명당 최대 1년, 분할 사용 2회까지 가능합니다.
- 육아기 근로시간 단축: 하루 1~5시간 단축 근무를 신청할 수 있습니다.
- 신청은 시작일 30일 전까지 인사팀에 서면으로 합니다.
""",

    "03_travel.md": """# 현대경제연구소 출장·경비 규정

※ 교육용 가상 문서입니다. 실제 기관의 규정과 무관합니다.

본 규정은 업무상 출장과 법인카드 사용에 적용됩니다. 담당: 재무팀

## 제1조 출장 신청

- 국내 출장: 출발 **3영업일 전**까지 그룹웨어 품의, 팀장 승인
- 해외 출장: 출발 **2주 전**까지 품의, 본부장 승인
- 해외 출장 품의서에는 일정표, 방문기관, 예상 경비를 반드시 적습니다.
- 출장 일정이 바뀌면 즉시 변경 품의를 올립니다.

## 제2조 국내 출장비

| 항목 | 기준 |
|------|------|
| 일비 | 1일 20,000원 |
| 식비 | 실비, 1끼 12,000원 한도 |
| 숙박비(서울) | 1박 120,000원 한도 |
| 숙박비(광역시) | 1박 100,000원 한도 |
| 숙박비(기타 지역) | 1박 80,000원 한도 |

- 당일 출장은 숙박비를 지급하지 않으며, 일비는 4시간 이상 출장 시 지급합니다.
- 숙박비 한도를 넘는 금액은 본인 부담입니다.
- 친척·지인 집에 묵는 경우 1박 30,000원을 정액 지급합니다.

## 제3조 교통비

- 철도: KTX·SRT 일반실 기준. 특실 이용 시 차액은 본인 부담입니다.
- 항공(국내): 제주 및 편도 4시간 이상 거리만 인정, 일반석 기준
- 자가용: km당 250원과 통행료를 지급합니다. 주차비는 실비로 인정합니다.
- 택시: 대중교통이 없는 구간이거나 22시 이후에만 인정됩니다.

## 제4조 해외 출장비

| 항목 | 북미·유럽 | 아시아 | 기타 지역 |
|------|-----------|--------|-----------|
| 일비 | USD 60 | USD 50 | USD 50 |
| 숙박비(1박 한도) | USD 200 | USD 150 | USD 150 |

- 항공권: 이코노미석 (임원은 비행 8시간 이상 시 비즈니스석)
- 항공권은 재무팀 지정 여행사를 통해 예약해야 합니다.
- 해외 여행자보험은 연구소가 일괄 가입합니다.
- 해외에서 법인카드 사용 시 환율은 카드사 청구 환율을 적용합니다.

## 제5조 경비 정산

- 출장 복귀 후 **7영업일 이내**에 영수증을 첨부해 정산합니다.
- 기한을 넘기면 재무팀 승인 후에만 정산이 가능합니다.
- 영수증을 분실한 경우 '지출증빙 확인서'를 작성하며, 건당 30,000원까지만 인정됩니다.
- 개인카드로 결제한 경비는 정산 승인 후 다음 급여일에 지급합니다.

## 제6조 법인카드 사용 제한

다음은 법인카드로 결제할 수 없습니다.
- 개인 용품, 상품권, 유흥업소
- 23시 이후 결제 (출장 중 불가피한 교통비 제외)
- 주류 단독 결제 (공식 회식 제외)
- 주말·공휴일 결제 (출장·행사 등 사전 승인 건 제외)

## 제7조 회식·접대비

- 팀 회식비: 분기별 1인당 30,000원
- 외부 접대비: 1인당 50,000원 한도, 초과 시 사전 본부장 승인 필요
- 접대비 정산 시 참석자 명단과 소속 기관을 반드시 적습니다.
- 공무원 등 청탁금지법 적용 대상자 접대 시 1인당 30,000원을 넘을 수 없습니다.
""",

    "04_security.md": """# 현대경제연구소 정보보안 규정

※ 교육용 가상 문서입니다. 실제 기관의 규정과 무관합니다.

연구소의 연구 성과와 고객 정보를 보호하기 위한 규정입니다. 담당: IT팀(정보보안파트)

## 제1조 문서 보안등급

| 등급 | 설명 | 외부 반출 |
|------|------|-----------|
| 대외비 | 미발표 연구결과, 경제전망 수치, 고객사 자료 | 금지 |
| 사내한 | 내부 회의자료, 업무 매뉴얼 | 팀장 승인 시 가능 |
| 공개 | 발간된 보고서, 보도자료 | 가능 |

- 문서 작성자는 작성 시 반드시 보안등급을 표시해야 합니다.
- 대외비 문서 출력 시 워터마크(출력자 사번·일시)가 자동으로 찍힙니다.
- 대외비 문서는 사내망에서만 열람할 수 있으며, 재택근무 중에는 열람할 수 없습니다.

## 제2조 엠바고(보도 유예)

- 경제전망 수치는 **공식 발표 전 외부에 언급할 수 없습니다.**
- 가족·지인과의 대화, 개인 SNS도 외부 언급에 해당합니다.
- 언론 문의는 개인이 답하지 말고 반드시 **홍보팀(내선 2400)** 으로 연결합니다.
- 엠바고 위반은 징계 사유입니다.

## 제3조 외부 생성형 AI 사용

- ChatGPT 등 외부 AI에는 **공개 등급 자료만** 입력할 수 있습니다.
- 대외비 자료, 미발표 전망치, 고객사 정보, 개인정보 입력은 **금지**입니다.
- 대외비·사내한 자료는 사내 AI 서비스 **'현경 어시스턴트'** 로만 처리합니다.
- AI가 만든 결과물을 보고서에 쓸 때는 반드시 사실 확인을 거치고, 출처를 밝힙니다.

## 제4조 저장매체·기기

- 개인 USB 사용은 금지되며, IT팀이 지급한 보안 USB만 사용할 수 있습니다.
- 보안 USB는 사내 PC에서만 읽히며, 반출 시 자동 암호화됩니다.
- 업무용 노트북을 외부로 가져갈 때는 그룹웨어에서 사전에 반출 신청을 합니다.
- 해외 출장 시 노트북 반출은 팀장 승인 외에 정보보안파트 승인이 추가로 필요합니다.
- 노트북·보안 USB 분실 시 **24시간 이내** IT헬프데스크(내선 1234)에 신고합니다.

## 제5조 계정·비밀번호

- 비밀번호는 10자 이상, 영문·숫자·특수문자 조합이어야 합니다.
- 90일마다 변경해야 하며, 최근 3회 사용한 비밀번호는 다시 쓸 수 없습니다.
- 5회 연속 틀리면 계정이 잠기며, IT헬프데스크(내선 1234)로 해제를 요청합니다.
- 계정은 본인만 사용해야 하며, 동료와 공유하면 안 됩니다.

## 제6조 이메일·메신저

- 외부로 첨부파일을 보낼 때는 자동으로 팀장 결재가 요청됩니다.
- 출처가 불분명한 메일의 링크나 첨부파일은 열지 말고 IT헬프데스크에 신고합니다.
- 개인 이메일·개인 메신저로 업무 파일을 보내는 것은 금지입니다.

## 제7조 클린데스크

- 퇴근 시 책상 위에 서류를 두지 않고, 대외비 문서는 잠금 서랍에 보관합니다.
- 자리를 비울 때는 PC 화면을 잠급니다. (Windows 키 + L)
- 정보보안파트는 월 1회 불시 점검을 합니다.
""",

    "05_welfare.md": """# 현대경제연구소 복리후생 안내

※ 교육용 가상 문서입니다. 실제 기관의 규정과 무관합니다.

직원의 건강과 성장을 지원하는 제도입니다. 담당: 인사팀(복리후생), 총무팀(시설)

## 1. 자기계발비

- 연 120만원 한도로 지원합니다. (입사 첫해는 입사 월부터 월할 계산)
- 대상: 어학, 통계·데이터 분석, 경제·금융 관련 강좌 및 도서
- 연구 관련 학회 참가비는 자기계발비와 별도로 업무 경비로 처리합니다.
- 영수증을 첨부해 그룹웨어 > 복리후생 메뉴에서 신청하며, 분기별로 지급합니다.
- 미사용 금액은 다음 해로 이월되지 않습니다.

## 2. 학위 취득 지원

- 근속 2년 이상 연구직은 석·박사 과정 학비의 50%를 지원받을 수 있습니다.
- 지원을 받은 경우 학위 취득 후 2년간 의무 근무해야 하며,
  그 전에 퇴직하면 지원금을 반환해야 합니다.

## 3. 건강검진

- 전 직원 연 1회 종합건강검진을 지원합니다. (지정 병원 3곳 중 선택)
- 만 40세 이상은 위·대장 내시경, 뇌 MRI 중 1개 항목을 추가로 선택할 수 있습니다.
- 배우자는 2년에 1회 기본 검진을 지원합니다.
- 검진 당일은 공가(유급)로 처리합니다.

## 4. 사내식당·카페

- 사내식당(본관 지하 1층): 조식 08:00~09:00, 중식 11:30~13:00
- 식대는 1끼 3,000원이며, 급여에서 공제됩니다. 조식은 무료입니다.
- 사내 카페(1층): 08:00~18:00, 직원 50% 할인

## 5. 주차 지원

- 사내 주차장은 임원, 임산부, 장애인 직원에게 우선 배정합니다.
- 일반 직원은 추첨으로 분기별 배정하며, 월 50,000원을 급여에서 공제합니다.
- 주차 신청은 분기 시작 2주 전까지 총무팀(내선 2300)에 합니다.
- 배정받지 못한 직원은 대중교통 이용을 권장하며, 별도 교통비는 지급하지 않습니다.

## 6. 휴양시설

- 제휴 콘도(강원·제주·경주)를 연 2회, 1회 최대 2박까지 이용할 수 있습니다.
- 성수기(7~8월, 12~2월)에는 추첨으로 배정합니다.
- 숙박비의 70%를 연구소가 지원합니다.

## 7. 경조 지원

- 경조휴가와 경조금은 근무·휴가 규정 제5조를 따릅니다.
- 직원 본인 사망 시 유가족에게 위로금 1,000만원을 지급합니다.
- 화환·조화는 인사팀이 대표로 보냅니다.
""",

}

# TODO: "hri_docs" 폴더 만들기 (이미 있어도 오류 안 나게)
os.makedirs("____", exist_ok=True)

# TODO: documents 딕셔너리에서 (파일명, 내용)을 하나씩 꺼내기
for filename, content in documents.____():
    with open(f"hri_docs/{filename}", "w", encoding="utf-8") as f:
        f.write(content)

print("저장한 파일:")
for filename, content in documents.items():
    print(f" - {filename}  ({len(content)}자)")

### 셀 9 — 폴더 전체를 한 번에 불러오기 (DirectoryLoader)

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "____",                 # TODO: 폴더 이름
    glob="____",            # TODO: .md 파일 전부를 뜻하는 패턴
    loader_cls=____,        # TODO: 파일 하나를 읽을 때 쓸 로더 클래스
    loader_kwargs={"encoding": "utf-8"},
)
all_docs = loader.load()

print("불러온 문서 수:", len(all_docs))
for d in sorted(all_docs, key=lambda d: d.metadata["source"]):
    print(f" - {os.path.basename(d.metadata['source'])}  ({len(d.page_content)}자)")

### 셀 10 — 규정 5개를 청크로 쪼개기

규정은 조항 하나가 200~400자 정도라서 `chunk_size=300`, `chunk_overlap=30`으로 시작합니다.

In [ ]:
# TODO: chunk_size 300, chunk_overlap 30
splitter = RecursiveCharacterTextSplitter(chunk_size=____, chunk_overlap=____)

# TODO: all_docs를 쪼개기
chunks = ____

print("전체 청크 개수:", len(chunks))
print()
print("───[ 첫 번째 청크 ]───")
print("본문:", chunks[0].page_content[:120], "...")
print("꼬리표:", chunks[0].metadata)

### 셀 11 — 꼬리표 달기: 분류(category) + 담당부서(department)

나중에 "보안 질문엔 보안 규정만 검색"하고, 답변에 "문의: IT팀"을 붙이기 위해 꼬리표를 2개 답니다.

In [ ]:
category_map = {
    "01_faq.md":      "faq",
    "02_leave.md":    "leave",
    "03_travel.md":   "____",      # TODO
    "04_security.md": "____",      # TODO
    "05_welfare.md":  "welfare",
}
department_map = {
    "faq":      "인사팀",
    "leave":    "인사팀",
    "travel":   "____",            # TODO: 출장·경비 담당 부서
    "security": "____",            # TODO: 정보보안 담당 부서
    "welfare":  "인사팀·총무팀",
}

for c in chunks:
    filename = os.path.basename(c.metadata["source"])
    c.metadata["category"] = category_map[filename]
    # TODO: category를 이용해 department 꼬리표 달기
    c.metadata["department"] = ____

from collections import Counter
print("분류별 청크 수:", dict(Counter(c.metadata["category"] for c in chunks)))
print()
print("예시 청크 꼬리표:", chunks[0].metadata)

> 🌱 이제 청크마다 `source`(파일), `category`(분류), `department`(담당부서)가 있습니다.

### 셀 12 — 쪼갠 효과 확인: 질문에 필요한 청크만 골라보기

김연구 씨 질문: **"신입인데 재택근무 할 수 있나요?"**

아직 의미 검색은 배우지 않았으니, 우선 **단어(키워드)** 로 청크를 골라봅니다.
그리고 **통째로 넣을 때와 글자 수**를 비교합니다.

In [ ]:
keyword = "재택"

# TODO: 본문(page_content)에 keyword가 들어간 청크만 고르기
found = [c for c in chunks if ____ in c.____]

total_chars = sum(len(d.page_content) for d in all_docs)
found_chars = sum(len(c.page_content) for c in found)

print(f"'{keyword}'이(가) 들어간 청크: {len(found)}개")
for c in found:
    print(f"\n[{c.metadata['category']} / 문의: {c.metadata['department']}]")
    print(c.page_content[:150], "...")

print("\n───────────────")
print(f"통째로 넣으면: {total_chars}자")
print(f"고른 청크만:   {found_chars}자  (약 {found_chars / total_chars:.0%})")

> 💡 필요한 청크만 주면 AI가 읽을 양이 크게 줄어듭니다.
> 재택근무 조항이 **근무·휴가 규정**과 **정보보안 규정** 양쪽에 걸쳐 있다는 것도 보이죠?
>
> ⚠️ 단, 키워드 검색은 "집에서 일해도 돼요?"처럼 **다른 단어**로 물으면 못 찾습니다. → 다음 시간 **임베딩(의미 검색)** 으로 해결합니다.

### 셀 13 — 표(table)는 어떻게 잘렸을까?

출장·경비 규정의 **해외 출장비 표**가 청크 경계에서 잘리지 않았는지 확인합니다.

In [ ]:
# TODO: category가 "travel"인 청크만 고르기
travel_chunks = [c for c in chunks if c.metadata["____"] == "____"]

for i, c in enumerate(travel_chunks):
    if "USD" in c.page_content:
        print(f"───[ travel 청크 {i} ]───")
        print(c.page_content)
        print()

> 🤔 표의 머리글(`| 항목 | 북미·유럽 | ...`)과 내용 행이 한 청크에 같이 있나요? 머리글이 떨어지면 "USD 200이 어느 지역 기준인지" AI가 알 수 없습니다.

# 5️⃣ 단계 — 미션

### 셀 14 — 미션 ①: 김연구 씨 질문에 맞는 청크 찾기

질문마다 **어떤 키워드로 찾을지** 정해서 청크를 찾아보세요.

| 김연구 씨 질문 | 키워드 예시 | 정답 위치 |
|----------------|-------------|-----------|
| 부산 출장 숙박비는 얼마까지? | `광역시` | travel |
| ChatGPT에 보고서 초안 넣어도 돼요? | `생성형 AI` | security |
| 자기계발비로 통계 강의 들을 수 있나요? | `자기계발비` | welfare |
| 회의실은 몇 시간까지 예약돼요? | `회의실` | faq |

In [ ]:
def find_chunks(keyword):
    """keyword가 들어간 청크를 찾아 출처·담당부서와 함께 보여준다."""
    found = [c for c in chunks if keyword in c.page_content]
    print(f"🔎 '{keyword}' → {len(found)}개")
    for c in found:
        print(f"   [{c.metadata['category']} / 문의: {c.metadata['department']}] {c.page_content[:60]!r}...")
    print()

# TODO: 표를 보고 키워드 4개를 넣어보세요. 내가 만든 질문의 키워드도 추가해 보세요.
for kw in ["____", "____", "____", "____"]:
    find_chunks(kw)

### 셀 15 — 미션 ②: chunk_size를 바꿔 표 지키기

셀 13에서는 300자로 잘라 표가 지켜졌습니다. 그럼 **더 작게** 자르면 어떻게 될까요?
크기별로 청크 수와 표가 지켜지는지를 함께 봅니다.

In [ ]:
# TODO: 5가지 크기를 넣어보세요 (예: 100, 150, 200, 300, 500)
for size in [____, ____, ____, ____, ____]:
    sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=30)
    cs = sp.split_documents(all_docs)
    # 해외 출장비 표 머리글과 'USD 200' 행이 같은 청크에 있는지 확인
    together = any("북미·유럽" in c.page_content and "USD 200" in c.page_content for c in cs)
    print(f"chunk_size={size:>3} → 청크 {len(cs):>3}개 | 표 머리글+내용 함께: {'✅' if together else '❌'}")

> 💡 150자 이하에서는 표 머리글과 내용이 떨어집니다(❌). 크게 자르면 표는 지켜지지만 청크 하나에 관계없는 내용이 섞입니다. 이 균형을 잡는 게 실무 감각입니다.

> ✍️ 내가 고른 크기: 

### 셀 16 — (도전) 조항 제목 기준으로 쪼개기

규정은 `## 제3조 연차휴가`처럼 **제목**으로 나뉘어 있습니다.
`MarkdownHeaderTextSplitter`를 쓰면 글자 수가 아니라 **조항 단위**로 자르고,
조항 제목이 꼬리표로 붙습니다. → 답변에 "(근거: 제3조 연차휴가)"를 붙이기 좋습니다.

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

md_splitter = MarkdownHeaderTextSplitter(
    # TODO: "#"은 "규정명", "##"은 "조항"으로 꼬리표 이름 붙이기
    headers_to_split_on=[("#", "____"), ("____", "조항")]
)

leave_doc = [d for d in all_docs if d.metadata["source"].endswith("02_leave.md")][0]
article_chunks = md_splitter.split_text(leave_doc.page_content)

print("조항 단위 청크 수:", len(article_chunks))
for c in article_chunks:
    print(f" - {c.metadata.get('조항', '(머리말)')}  ({len(c.page_content)}자)")

print("\n───[ 예시 ]───")
print("꼬리표:", article_chunks[3].metadata)
print(article_chunks[3].page_content[:200])

> 💡 조항마다 길이가 들쭉날쭉하죠? 실무에서는 **제목으로 먼저 자르고 → 너무 긴 조항만 다시 글자 수로 자르는** 2단계 방식을 많이 씁니다.

---
# 🎓 정리

| 단계 | 한 일 | 핵심 |
|------|-------|------|
| 1️⃣ 이론 | 통째로의 한계 → 쪼개기 필요 | 불러오기 + 쪼개기 |
| 2️⃣ 기본 | `loader.load()` / `split_documents()` | 본문 + 꼬리표 |
| 3️⃣ 응용 | size·overlap 바꿔보기 | 적당한 크기 감각 |
| 4️⃣ 실습 | 사내규정 5개 → 청크 + 분류·담당부서 꼬리표 | 필요한 청크만 고르면 글자 수 ↓ |
| 5️⃣ 미션 | 키워드로 찾기, 표 지키기, 조항 단위 쪼개기 | 키워드 검색의 한계 → 다음 시간 |

**한 문장 요약**: 사내규정을 **불러와서** 조항 크기로 **잘게 쪼개고**, 출처·분류·담당부서 **꼬리표**를 달아둔다.

**다음 시간**: "집에서 일해도 돼요?"처럼 단어가 달라도 재택근무 조항을 찾도록, **임베딩과 벡터 데이터베이스**로 의미 검색을 합니다.

## 🤔 생각해보기

1. 셀 12에서 "재택"으로 찾지 못하는 질문을 3개 만들어 보세요.
2. 규정이 개정되어 `02_leave.md`만 바뀌었다면, 전체를 다시 쪼개야 할까요? 꼬리표를 어떻게 활용할 수 있을까요?
3. 보안 규정처럼 **틀리면 안 되는 문서**는 다른 문서보다 크게 자를까요, 작게 자를까요? 이유는?

> ✍️ 메모:
> - 1번:
> - 2번:
> - 3번: